# 24788 Project: Reproduce Results
**Authors:** Zixuan Wang & Yiman Wu

This notebook reproduces the test set metrics (MAE in meV) using pre-trained checkpoints for GCN, SchNet, and DimeNet.

In [ ]:
# Install Dependencies
!pip install torch-geometric
!pip install torch-cluster torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.5.0+cu121.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.2 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.5.0+cu121.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 13.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 84.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 61.1 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, SchNet, DimeNet
import os

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

TARGET_IDX = 4  # HOMO-LUMO gap

# 1. Load Dataset and Normalization Constants
dataset = QM9(root='./data/QM9')
all_targets = dataset.data.y[:, TARGET_IDX]
TARGET_MEAN = all_targets.mean().item()
TARGET_STD  = all_targets.std().item()

# Apply normalization to test set
dataset.data.y[:, TARGET_IDX] = (dataset.data.y[:, TARGET_IDX] - TARGET_MEAN) / TARGET_STD
test_dataset = dataset[120000:130000]
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Loaded {len(test_dataset)} samples for testing.")

/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_scatter/_version_cuda.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_cluster/_version_cuda.so
  import torch_geometric.typing
/usr/local/lib/python3.12/dist-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: Could not load this library: /usr/local/lib/python3.12/dist-packages/torch_sparse/_version_cuda.so
  import torch_geometric.typing


Using device: cpu


Extracting data/QM9/raw/qm9_v3.zip
Processing...
Using a pre-processed version of the dataset. Please install 'rdkit' to alternatively process the raw data.
Done!


Loaded 10000 samples for testing.


/tmp/ipykernel_17384/1746851525.py:15: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  all_targets = dataset.data.y[:, TARGET_IDX]
/tmp/ipykernel_17384/1746851525.py:20: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  dataset.data.y[:, TARGET_IDX] = (dataset.data.y[:, TARGET_IDX] - TARGET_MEAN) / TARGET_STD


## 2. Model Definitions

In [ ]:
# --- GCN Baseline ---
class GCNBaseline(nn.Module):
    def __init__(self, in_channels=11, hidden_channels=64, num_layers=3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()
        self.convs.append(GCNConv(in_channels, hidden_channels))
        self.bns.append(nn.BatchNorm1d(hidden_channels))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
        self.mlp = nn.Sequential(nn.Linear(hidden_channels, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index).relu()
            x = bn(x)
        x = global_mean_pool(x, batch)
        return self.mlp(x).squeeze(-1)

# --- SchNet ---
class SchNetModel(nn.Module):
    def __init__(self, hidden_channels=128, num_interactions=6):
        super().__init__()
        self.embedding = nn.Embedding(100, hidden_channels)
        self.dist_embedding = nn.Sequential(
            nn.Linear(1, hidden_channels), nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
        )
        self.interactions = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels), nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
            ) for _ in range(num_interactions)
        ])
        self.output = nn.Sequential(
            nn.Linear(hidden_channels, 64), nn.ReLU(), nn.Linear(64, 1)
        )

    def forward(self, data):
        z = data.x[:, 0].long()
        pos, edge_index, batch = data.pos, data.edge_index, data.batch
        h = self.embedding(z)
        row, col = edge_index
        dist = (pos[row] - pos[col]).norm(dim=-1, keepdim=True)
        edge_feat = self.dist_embedding(dist)
        for interaction in self.interactions:
            agg = torch.zeros_like(h)
            msg = h[col] * edge_feat
            agg.scatter_add_(0, row.unsqueeze(-1).expand_as(msg), msg)
            h = h + interaction(agg)
        h = global_mean_pool(h, batch)
        return self.output(h).squeeze(-1)




# --- DimeNet ---
class DimeNetModel(nn.Module):
    def __init__(self, hidden_channels=128, num_interactions=6):
        super().__init__()
        self.embedding = nn.Embedding(100, hidden_channels)
        self.dist_embedding = nn.Sequential(
            nn.Linear(1, hidden_channels), nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
        )
        self.angle_embedding = nn.Sequential(
            nn.Linear(1, hidden_channels), nn.ReLU(),
            nn.Linear(hidden_channels, hidden_channels),
        )
        self.interactions = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_channels, hidden_channels), nn.ReLU(),
                nn.Linear(hidden_channels, hidden_channels),
            ) for _ in range(num_interactions)
        ])
        self.output = nn.Sequential(
            nn.Linear(hidden_channels, 64), nn.ReLU(), nn.Linear(64, 1)
        )

    def forward(self, data):
        z = data.x[:, 0].long()
        pos, edge_index, batch = data.pos, data.edge_index, data.batch
        h = self.embedding(z)
        row, col = edge_index

        diff = pos[row] - pos[col]
        dist = diff.norm(dim=-1, keepdim=True)
        edge_feat = self.dist_embedding(dist)

        diff_norm = diff / dist.clamp(min=1e-8)  # [num_edges, 3]

        avg_dir = torch.zeros(h.shape[0], 3, device=h.device)
        avg_dir.scatter_add_(0, row.unsqueeze(-1).expand(-1, 3), diff_norm)
        count = torch.zeros(h.shape[0], 1, device=h.device)
        count.scatter_add_(0, row.unsqueeze(-1), torch.ones(row.shape[0], 1, device=h.device))
        avg_dir = avg_dir / count.clamp(min=1e-8)  # [num_nodes, 3]

        cos_angle = (diff_norm * avg_dir[row]).sum(dim=-1, keepdim=True).clamp(-1, 1)
        angle_feat_edge = self.angle_embedding(cos_angle)  # [num_edges, hidden]

        angle_feat = torch.zeros_like(h)
        angle_feat.scatter_add_(0, row.unsqueeze(-1).expand_as(angle_feat_edge), angle_feat_edge)

        for interaction in self.interactions:
            agg = torch.zeros_like(h)
            msg = h[col] * edge_feat
            agg.scatter_add_(0, row.unsqueeze(-1).expand_as(msg), msg)
            h = h + interaction(agg + angle_feat)

        h = global_mean_pool(h, batch)
        return self.output(h).squeeze(-1)

In [5]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_mae = 0
    for batch in loader:
        batch = batch.to(DEVICE)
        pred = model(batch) * TARGET_STD + TARGET_MEAN
        target = batch.y[:, TARGET_IDX] * TARGET_STD + TARGET_MEAN
        total_mae += (pred - target).abs().sum().item()
    return total_mae / len(loader.dataset)

models_to_test = {
    'GCN': GCNBaseline(),
    'SchNet': SchNetModel(),
    'DimeNet': DimeNetModel()
}

print(f"\n{'Model':<10} | {'Test MAE (meV)':<15}")
print("-" * 30)

for name, model in models_to_test.items():
    ckpt_path = f'best_{name}.pt'
    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        model.to(DEVICE)
        mae = evaluate(model, test_loader)
        print(f"{name:<10} | {mae*1000:>15.1f}")
    else:
        print(f"{name:<10} | Checkpoint not found at {ckpt_path}")


Model      | Test MAE (meV) 
------------------------------
GCN        |           300.3
SchNet     |           158.3
DimeNet    |           120.3
